# DENTEX End-to-End Workflow

This notebook uses the copied `DentexSegAndDet` codebase and is structured into:

1. Preprocessing
2. Augmentation
3. Training
4. Testing


## 0) Environment Setup

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
CODE_ROOT = PROJECT_ROOT / 'DentexSegAndDet'
DATA_ROOT = PROJECT_ROOT / 'dentex_dataset'

print(f'Project root: {PROJECT_ROOT}')
print(f'Code root: {CODE_ROOT}')
print(f'Data root: {DATA_ROOT}')


In [ ]:
# Run this in Kaggle/Colab if dependencies are missing.
# %pip install -q -r requirements-colab.txt


## 1) Preprocessing

In [ ]:
from huggingface_hub import snapshot_download

DATA_ROOT.mkdir(parents=True, exist_ok=True)
snapshot_path = snapshot_download(
    repo_id='ibrahimhamamci/DENTEX',
    repo_type='dataset',
    local_dir=str(DATA_ROOT),
    local_dir_use_symlinks=False,
)
print(f'Downloaded to: {snapshot_path}')


In [ ]:
import sys

sys.path.insert(0, str(CODE_ROOT))
os.chdir(CODE_ROOT)

import process_dataset

# The original pipeline expects ./dentex_dataset relative to DentexSegAndDet.
if not (CODE_ROOT / 'dentex_dataset').exists():
    os.symlink(DATA_ROOT, CODE_ROOT / 'dentex_dataset')

process_dataset.process_coco_quadrant()
process_dataset.process_coco_enumeration32()
process_dataset.process_coco_disease()
process_dataset.process_coco_disease_all()
process_dataset.process_yolo_disease_all()
process_dataset.process_yolo_disease()
process_dataset.process_seg_enumeration32()
# process_dataset.process_seg_enumeration9('results/enumeration_dataset_quadrant_predictions.json')


## 2) Augmentation

In [ ]:
# Baseline training scripts already include transforms for training.
# Leave this as False for baseline reproducibility.
USE_EXTRA_AUGMENTATION = False

if USE_EXTRA_AUGMENTATION:
    print('Add custom augmentation pipeline here before training.')
else:
    print('Using built-in augmentation from DentexSegAndDet scripts.')


## 3) Training

In [ ]:
# Run one block at a time and adjust paths to pretrained checkpoints.
# !python train_diffdet.py --output-dir output_diffdet_quadrant --config-file configs/diffdet/diffdet.dentex.swinbase.quadrant.yaml MODEL.WEIGHTS checkpoints/swin_base_patch4_window7_224_22k.pkl
# !python train_dino.py --output_dir output_dino_res50_enum32 -c configs/dino/DINO_4scale_cls32.py --coco_path dentex_dataset/coco/enumeration32 --options dn_scalar=100 embed_init_tgt=TRUE dn_label_coef=1.0 dn_bbox_coef=1.0 use_ema=False dn_box_noise_scale=1.0 --pretrain_model_path checkpoints/dino_pretrained_checkpoint0033_4scale.pth --finetune_ignore label_enc.weight class_embed
# !python train_yolo.py
# !python train_unet.py --output_dir output_unet_enum32 --dataset_dir dentex_dataset/segmentation/enumeration32 --num_classes 32 --model seunet


## 4) Testing

In [ ]:
# Update checkpoint paths in predict.py before running.
# !python predict.py
